# Evaluation

**Goal:** the single, authoritative readout of model quality on the **held-out test set** — 294 employees at the real-world ~16% attrition rate that no stage before this one was allowed to learn from.

Phases 5â€“6 deliberately kept the test set sealed: every earlier test number was flagged *"indicative only; authoritative evaluation deferred to Phase 7."* This stage opens it. It runs a **full bake-off** — all four tuned models scored side by side — reports the complete metric suite (recall, precision, F1, F2, ROC-AUC, PR-AUC, confusion matrix) the project objectives call for, and confirms the model to deploy.

**One leakage caveat, stated up front.** Each model's decision threshold was fixed on leakage-free out-of-fold (OOF) probabilities in Phase 6 and is **not** re-tuned here, so the comparison is fair. But picking the *deployed* model purely on a 294-row test sample risks overfitting that sample, so the final rule is **conservative**: confirm the Phase 6 OOF leader unless the test set contradicts it by a material margin.

## 1. Setup

The metric toolkit (`recall`, `precision`, `f1`, `fbeta`, `roc_auc`, `average_precision`, plus `confusion_matrix`, `classification_report`, and the `roc_curve` / `precision_recall_curve` plot inputs) alongside `matplotlib`/`seaborn`. `RANDOM_STATE = 42` matches every earlier stage. Two paths: `PROC = Path('processed')` (the `processed/` folder sits inside `notebooks/`) and `FIG = Path('outputs')`, created here so every figure can be saved as a PNG for reuse in reports and slides.

In [ ]:
# Cell 1 — Imports & setup
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    recall_score, precision_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', context='notebook')

RANDOM_STATE = 42
PROC = Path('processed')          # processed/ lives inside notebooks/
FIG = Path('outputs')             # figures saved here for reuse in reports/slides
FIG.mkdir(exist_ok=True)

## 2. Load the test set, tuned models, and prior results

The four `models/*_tuned.joblib` estimators (already refit on the full training set by `GridSearchCV`) plus the artifacts that frame the comparison:

- `tuning_selection.joblib` — the Phase 6 OOF leader and its threshold.
- `tuning_results.parquet` — per-model **tuned thresholds** (each model's max-OOF-F2 operating point) and OOF metrics.
- `threshold_sweep.parquet`, `cv_results.parquet` — the OOF sweep and the Phase 5 CV baseline, for the journey table.

`tuned_threshold` maps each model to the threshold it earned in Phase 6 — these are applied verbatim, never re-fit on test.

In [ ]:
# Cell 2 — Load the held-out test set, the tuned models, and Phase 5/6 results
X_test = pd.read_parquet(PROC / 'X_test.parquet')
y_test = pd.read_parquet(PROC / 'y_test.parquet')['Attrition']

# Phase 6 selection metadata (OOF leader + its tuned threshold) and the per-model summary.
tuning_selection = joblib.load(PROC / 'tuning_selection.joblib')
tuning_results = pd.read_parquet(PROC / 'tuning_results.parquet')
threshold_sweep = pd.read_parquet(PROC / 'threshold_sweep.parquet')
baseline_cv = pd.read_parquet(PROC / 'cv_results.parquet')   # Phase 5 CV baseline

# Display name -> slug used for the saved estimator files.
SLUG = {
    'LogReg (SMOTE)': 'logreg_smote',
    'RandomForest': 'random_forest',
    'XGBoost': 'xgboost',
    'LightGBM': 'lightgbm',
}
# Each model carries the threshold chosen on OOF data in Phase 6 (max-OOF-F2).
tuned_threshold = dict(zip(tuning_results['model'], tuning_results['threshold']))

MODELS = {
    name: joblib.load(PROC / 'models' / f'{slug}_tuned.joblib')
    for name, slug in SLUG.items()
}

oof_leader = tuning_selection['leader']
print(f'X_test {X_test.shape} | attrition {y_test.mean():.3f}')
print(f'Loaded {len(MODELS)} tuned models: {", ".join(MODELS)}')
print(f'Phase 6 OOF leader: {oof_leader} @ threshold {tuning_selection["threshold"]:.2f}')

## 3. Helpers

`evaluate(y_true, proba, threshold)` returns the full positive-class ("Leave") metric suite plus the four confusion-matrix counts (`tn`, `fp`, `fn`, `tp`) in one dict, so the bake-off table, the heatmaps, and the persisted metadata all read from the same source. `savefig(name)` writes the current figure to `outputs/<name>.png` at 150 dpi. Both are defined inline because helpers do not carry across stage notebooks.

In [ ]:
# Cell 3 — Helpers
# Defined inline because helpers do not carry across stage notebooks.
def evaluate(y_true, proba, threshold):
    """Full positive-class ('Leave') metric suite + confusion counts at a probability threshold."""
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'threshold': threshold,
        'recall': recall_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred),
        'f2': fbeta_score(y_true, pred, beta=2),
        'roc_auc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'accuracy': (tp + tn) / (tp + tn + fp + fn),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def savefig(name):
    """Save the current figure to outputs/<name>.png for reuse in reports/slides."""
    plt.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight')

## 4. The bake-off

Every tuned model scores the test set at **its own** Phase 6 threshold, ranked by the project rule — **recall first, F1 as tiebreak** (catching leavers is the priority; F1 breaks ties). The table reports all metrics side by side with raw confusion counts so the precision/recall trade-off is legible, and names the test-set winner. Per-model probabilities are cached in `test_proba` for the curve plots that follow.

In [ ]:
# Cell 4 — Bake-off: every tuned model scored on the held-out test set
# Each model is evaluated at the threshold fixed on OOF data in Phase 6 (never re-tuned
# on test). Ranked by the project rule: recall first, F1 as tiebreak.
test_proba = {name: est.predict_proba(X_test)[:, 1] for name, est in MODELS.items()}

eval_rows = [{'model': name, **evaluate(y_test, test_proba[name], tuned_threshold[name])}
             for name in MODELS]
eval_results = (
    pd.DataFrame(eval_rows)
    .sort_values(['recall', 'f1'], ascending=False)
    .reset_index(drop=True)
)
test_winner = eval_results.iloc[0]['model']

cols = ['model', 'threshold', 'recall', 'precision', 'f1', 'f2',
        'roc_auc', 'pr_auc', 'accuracy', 'tn', 'fp', 'fn', 'tp']
print(eval_results[cols].to_string(index=False))
print(f'\nTest-set winner (recall -> F1): {test_winner}')

## 5. Confusion matrices

A 2×2 grid, one heatmap per model at its tuned threshold, each titled with its recall and precision. Reading the bottom row — `fn` (leavers missed) vs `tp` (leavers caught) — is the most direct view of how well each model serves the retention goal. Saved as `confusion_matrices.png`.

### Confusion Matrices by Model

#### RandomForest (threshold: 0.49)
| | Predicted Stay | Predicted Leave |
|---|---|---|
| **Actual Stay** | 162 (TN) | 85 (FP) |
| **Actual Leave** | 8 (FN) | 39 (TP) |

**Metrics:** Recall = 0.830, Precision = 0.315

#### LogReg (SMOTE) (threshold: 0.50)
| | Predicted Stay | Predicted Leave |
|---|---|---|
| **Actual Stay** | 207 (TN) | 40 (FP) |
| **Actual Leave** | 16 (FN) | 31 (TP) |

**Metrics:** Recall = 0.660, Precision = 0.437

#### LightGBM (threshold: 0.31)
| | Predicted Stay | Predicted Leave |
|---|---|---|
| **Actual Stay** | 208 (TN) | 39 (FP) |
| **Actual Leave** | 19 (FN) | 28 (TP) |

**Metrics:** Recall = 0.596, Precision = 0.418

#### XGBoost (threshold: 0.42)
| | Predicted Stay | Predicted Leave |
|---|---|---|
| **Actual Stay** | 204 (TN) | 43 (FP) |
| **Actual Leave** | 20 (FN) | 27 (TP) |

**Metrics:** Recall = 0.574, Precision = 0.386

**Summary:** RandomForest achieves the highest recall (0.830 — catches 39 out of 47 leavers) but has more false positives (85). LogReg is more conservative with fewer false positives (40) but lower recall (0.660). In a business scenario, this might result in more employees getting flagged for leaving but value can be gained by this because even a low-risk employee now could become a higher risk one later on, if they are given overtime or made to travel or passed up for promotions.

In [ ]:
# Cell 5 — Confusion matrices for all four models (at their tuned thresholds)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, name in zip(axes.ravel(), MODELS):
    row = eval_results.loc[eval_results['model'] == name].iloc[0]
    cm = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'])
    ax.set_title(f"{name} @ {row['threshold']:.2f}\n"
                 f"recall {row['recall']:.2f} | precision {row['precision']:.2f}")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
fig.suptitle('Confusion matrices on the held-out test set', fontsize=13)
fig.tight_layout()
savefig('confusion_matrices')
plt.show()

## 6. ROC curves

All four ROC curves overlaid with their AUCs, against the chance diagonal. ROC-AUC measures threshold-independent rank-ordering of risk — useful for comparing the models' overall discriminative power regardless of operating point. Saved as `roc_curves.png`.

In [ ]:
# Cell 6 — ROC curves (all four models overlaid)
plt.figure(figsize=(7, 6))
for name in MODELS:
    fpr, tpr, _ = roc_curve(y_test, test_proba[name])
    auc = roc_auc_score(y_test, test_proba[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate (recall)')
plt.title('ROC curves — test set')
plt.legend(loc='lower right')
savefig('roc_curves')
plt.show()

## 7. Precisionâ€“Recall curves

The same four models as PR curves with their average-precision (PR-AUC) scores, against the positive-rate baseline (~0.16). Under heavy class imbalance PR-AUC is the more honest headline than ROC-AUC: it ignores the easy true negatives and focuses on performance on the minority "Leave" class we actually care about. Saved as `pr_curves.png`.

In [ ]:
# Cell 7 — Precision-Recall curves (all four models overlaid)
# PR-AUC is the more honest summary under 16% class imbalance than ROC-AUC.
plt.figure(figsize=(7, 6))
for name in MODELS:
    prec, rec, _ = precision_recall_curve(y_test, test_proba[name])
    ap = average_precision_score(y_test, test_proba[name])
    plt.plot(rec, prec, label=f'{name} (PR-AUC = {ap:.3f})')
plt.axhline(y_test.mean(), color='k', ls='--', lw=1,
            label=f'Baseline ({y_test.mean():.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curves — test set')
plt.legend(loc='upper right')
savefig('pr_curves')
plt.show()

## 8. Threshold trade-off for the winner

Recall, precision, F1, and F2 swept across thresholds 0.05â€“0.95 on the **winner's test probabilities**, with a vertical line at the Phase 6 threshold. This makes the operating-point decision explicit on unseen data: how much precision is sacrificed for each point of recall, and whether the chosen threshold still sits in a sensible place once we look at the test set. Saved as `threshold_tradeoff.png`.

In [ ]:
# Cell 8 — Threshold trade-off for the winner (on test probabilities)
# Shows how recall/precision/F1/F2 move as the operating point slides; the vertical
# line marks the threshold fixed in Phase 6.
winner_proba = test_proba[test_winner]
winner_t = tuned_threshold[test_winner]
grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

sweep = pd.DataFrame([
    {'threshold': t,
     'recall': recall_score(y_test, (winner_proba >= t).astype(int)),
     'precision': precision_score(y_test, (winner_proba >= t).astype(int), zero_division=0),
     'f1': f1_score(y_test, (winner_proba >= t).astype(int)),
     'f2': fbeta_score(y_test, (winner_proba >= t).astype(int), beta=2)}
    for t in grid
])

plt.figure(figsize=(8, 6))
for metric in ['recall', 'precision', 'f1', 'f2']:
    plt.plot(sweep['threshold'], sweep[metric], label=metric)
plt.axvline(winner_t, color='k', ls='--', lw=1, label=f'tuned threshold ({winner_t:.2f})')
plt.xlabel('Decision threshold')
plt.ylabel('Score')
plt.title(f'Threshold trade-off — {test_winner} (test set)')
plt.legend()
savefig('threshold_tradeoff')
plt.show()

## 9. The tuning journey

One row per model tracing recall and F1 through all three checkpoints — **Phase 5 CV â†’ Phase 6 OOF â†’ Phase 7 test**. Close OOF-vs-test values confirm the tuning gains generalise; a large drop from OOF to test would flag a model that overfit the training folds. This is the sanity check that the whole pipeline held together.

In [ ]:
# Cell 9 — The tuning journey: Phase 5 CV -> Phase 6 OOF -> Phase 7 test
# A growing OOF-vs-test gap would flag overfitting; close values mean the gains generalise.
journey = (
    eval_results[['model', 'recall', 'f1']]
    .rename(columns={'recall': 'test_recall', 'f1': 'test_f1'})
    .merge(tuning_results[['model', 'oof_recall', 'oof_f1']], on='model')
    .merge(baseline_cv[['model', 'recall', 'f1']]
           .rename(columns={'recall': 'cv_recall_p5', 'f1': 'cv_f1_p5'}), on='model')
)
journey = journey[['model', 'cv_recall_p5', 'oof_recall', 'test_recall',
                   'cv_f1_p5', 'oof_f1', 'test_f1']]
print(journey.sort_values('test_recall', ascending=False).to_string(index=False))

## 10. Final selection and business readout

The **conservative selection rule** in action: the Phase 6 OOF leader stays the deployed model unless the test winner beats it on recall by a material margin (≥ 0.05). This guards against re-optimising on a 294-row sample. The cell prints the `classification_report` for the confirmed model and translates its confusion matrix into business terms — how many real leavers are caught vs missed, and how many flagged employees are false alarms HR would follow up on needlessly. Where RandomForest (highest recall) and LogReg (best precision/F2 balance) diverge, the trade-off is stated so the choice is transparent.

In [ ]:
# Cell 10 — Final selection and business readout
# Conservative rule: keep the Phase 6 OOF leader as the deployed model unless the test
# set materially contradicts it under recall -> F1. Choosing a model on test performance
# risks overfitting the 294-row sample, so we confirm rather than re-optimise.
winner_row = eval_results.iloc[0]
oof_row = eval_results.loc[eval_results['model'] == oof_leader].iloc[0]

confirmed = oof_leader  # default: stick with the OOF leader
margin = winner_row['recall'] - oof_row['recall']
if test_winner != oof_leader and margin >= 0.05:
    confirmed = test_winner  # test set materially favours a different model

sel = eval_results.loc[eval_results['model'] == confirmed].iloc[0]
print(f'Phase 6 OOF leader : {oof_leader}')
print(f'Phase 7 test winner: {test_winner} (recall margin over OOF leader: {margin:+.3f})')
print(f'Confirmed deployed : {confirmed} @ threshold {sel["threshold"]:.2f}\n')

print(classification_report(
    y_test, (test_proba[confirmed] >= sel['threshold']).astype(int),
    target_names=['Stay', 'Leave'], digits=3))

leavers = int(sel['tp'] + sel['fn'])
print(f'Business readout for {confirmed}:')
print(f'  Of {leavers} actual leavers, {int(sel["tp"])} flagged, {int(sel["fn"])} missed '
      f'(recall {sel["recall"]:.0%}).')
print(f'  HR follows up on {int(sel["tp"] + sel["fp"])} employees; {int(sel["fp"])} are '
      f'false alarms (precision {sel["precision"]:.0%}).')

## 11. Persist artifacts

The bake-off table is saved as `evaluation_results.parquet`, and `final_evaluation.joblib` records the confirmed model, its threshold, the full test metrics, the confusion counts, the selection rule, and the leakage note. If the confirmed model differs from the Phase 6 leader, `models/final_model.joblib` is re-pointed to it so Phase 8 (interpretation) loads the right estimator; otherwise it is left untouched. The four PNGs already sit in `outputs/`.

In [ ]:
# Cell 11 — Persist evaluation artifacts
eval_results.to_parquet(PROC / 'evaluation_results.parquet', index=False)

joblib.dump({
    'selected_model': confirmed,
    'slug': SLUG[confirmed],
    'threshold': float(sel['threshold']),
    'oof_leader': oof_leader,
    'test_winner': test_winner,
    'selection_rule': 'confirm OOF leader unless test recall margin >= 0.05',
    'test_metrics': {k: float(sel[k]) for k in
                     ['recall', 'precision', 'f1', 'f2', 'roc_auc', 'pr_auc', 'accuracy']},
    'confusion': {k: int(sel[k]) for k in ['tn', 'fp', 'fn', 'tp']},
    'leakage_note': ('Thresholds were fixed on OOF data in Phase 6, not re-tuned on test. '
                     'The deployed model is confirmed, not re-selected, on test performance.'),
}, PROC / 'final_evaluation.joblib')

# Keep models/final_model.joblib aligned with the confirmed deployment choice.
if confirmed != oof_leader:
    joblib.dump(MODELS[confirmed], PROC / 'models' / 'final_model.joblib')
    print(f'final_model.joblib re-pointed to {confirmed} (was {oof_leader}).')
else:
    print(f'final_model.joblib unchanged ({oof_leader} confirmed).')

print(f'Saved evaluation_results.parquet and final_evaluation.joblib to {PROC.resolve()}')
print(f'Saved 4 figures to {FIG.resolve()}')